# 📘 Notebook 11: Big O Notation

### Course: *From Zero to Data Structures & Algorithms in Python*
**Instructor:** Ioannis Tsioulis

*Module C: Algorithms & Complexity · Notebook 1 of 4 in this module · (11 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain why we describe the speed of a program by **how it scales**, not by seconds
- Time a piece of code and plot the result
- Recognise **O(1)**, **O(n)** and **O(n²)** behaviour in an experiment and in code
- Explain what **O(log n)** and **O(n log n)** mean
- Determine the Big O of a function by reading its loops
- State the cost of the everyday operations on Python lists and dictionaries

> **Prerequisites:** Module A, especially Notebooks 03 (loops), 04 (functions) and 05 (lists and dictionaries).
>
> 🔭 **Why this notebook deserves your full attention:** everything that follows in this course, searching, sorting, stacks, linked lists, hash tables, trees, is judged with the tool introduced here. Whenever we compare two ways of doing something, the question will be: *what is its Big O?* We will never simply claim an answer. We will **measure it**.

## 1. Two programs, the same answer

### Intuition first
Two programs can both be correct and yet be worlds apart. One finishes instantly, the other makes you wait for hours. The difference rarely matters for ten items. It matters enormously for ten million.

So the interesting question is not *"how many seconds does it take?"* Seconds depend on the computer, on what else is running, on the programming language. The interesting question is:

> **When the input becomes 10 times bigger, how much longer does the program take?**

- Not at all? Excellent.
- 10 times longer? Reasonable.
- 100 times longer? Then a big enough input will bring the program to its knees.

**Big O notation** is the vocabulary for answering this question. We write the answer as a function of **n**, the size of the input.

### Our measuring tool
`time.perf_counter()` returns the current reading of a precise clock, in seconds. Reading it before and after a piece of code gives the time that code took.

In [ ]:
import time

start = time.perf_counter()

total = 0
for i in range(1_000_000):      # underscores make large numbers readable
    total += i

end = time.perf_counter()
print(f"The loop took {end - start:.4f} seconds.")

We will time many functions, so let us wrap the stopwatch in a function of its own. Notice that `measure` receives **another function** as an argument and calls it.

In [ ]:
def measure(function, argument):
    start = time.perf_counter()
    function(argument)
    return time.perf_counter() - start

## 2. O(1): constant time

### Intuition first
An operation is **O(1)** when it takes the same time **regardless of the size of the input**. Looking at the first page of a book takes the same effort whether the book has 50 pages or 5,000.

### The standard example: reading a list element by its index
Python knows exactly where element number `i` is stored in memory. It goes straight there, without looking at any other element.

In [ ]:
def get_middle_element(numbers):
    return numbers[len(numbers) // 2]

small_list = list(range(1_000))
large_list = list(range(1_000_000))

print(get_middle_element(small_list))
print(get_middle_element(large_list))

### Experiment: does the size of the list matter?
A single access is far too quick to time reliably, so for each list size we perform **one million** accesses and time them all together. If access is truly O(1), the total time should be about the same for every size.

In [ ]:
import random
import matplotlib.pyplot as plt

def one_million_accesses(numbers):
    n = len(numbers)
    for i in range(1_000_000):
        numbers[i % n]              # read an element; the index stays in range

sizes = [1_000, 10_000, 100_000, 1_000_000]
times = []

for n in sizes:
    numbers = list(range(n))
    elapsed = measure(one_million_accesses, numbers)
    times.append(elapsed)
    print(f"n = {n:>9,}: {elapsed:.4f} seconds")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(sizes, times, marker="o")
plt.ylim(0, max(times) * 2)
plt.xscale("log")
plt.title("O(1): one million list accesses")
plt.xlabel("Size of the list (n)")
plt.ylabel("Time (seconds)")
plt.grid(True)
plt.show()

### Observed result
The list grew **a thousand times** larger, and the time stayed essentially the same. The line is flat. Small differences are only noise from the computer doing other things at the same moment.

> 🧠 O(1) does not mean *fast*. It means *the time does not depend on n*. An operation that always takes one full second is O(1) too.

## 3. O(n): linear time

### Intuition first
An algorithm is **O(n)** when its time grows **in proportion** to the input. To read a whole book you must read every page: a book twice as long takes twice as long.

### The standard example: adding up a list
There is no shortcut. To know the total, every element must be visited once.

In [ ]:
def sum_list(numbers):
    total = 0
    for num in numbers:
        total += num
    return total

print(sum_list([1, 2, 3, 4, 5]))

### Experiment: doubling the input
We double the size of the list each time. If the function is O(n), the time should roughly **double** as well.

In [ ]:
sizes = [250_000, 500_000, 1_000_000, 2_000_000, 4_000_000]
times = []

for n in sizes:
    numbers = list(range(n))
    elapsed = measure(sum_list, numbers)
    times.append(elapsed)
    print(f"n = {n:>9,}: {elapsed:.4f} seconds")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(sizes, times, marker="o")
plt.title("O(n): summing a list")
plt.xlabel("Size of the list (n)")
plt.ylabel("Time (seconds)")
plt.grid(True)
plt.show()

### Observed result
The points lie on a **straight line**. Each doubling of `n` doubles the time. That straight line is the signature of linear time.

## 4. O(n²): quadratic time

### Intuition first
An algorithm is **O(n²)** when, for **each** element, it does something with **every** element. If everyone in a room shakes hands with everyone else, doubling the number of people makes four times as many handshakes.

### The standard example: a loop inside a loop
In Notebook 03 we noticed that nested loops **multiply**. The outer loop runs `n` times, and for each of those the inner loop runs `n` times: `n × n = n²` steps.

In [ ]:
def count_pairs(numbers):
    count = 0
    for a in numbers:
        for b in numbers:
            count += 1          # one step for every pair (a, b)
    return count

print(count_pairs([1, 2, 3]))          # 3 x 3 = 9 pairs
print(count_pairs(list(range(100))))   # 100 x 100 = 10,000 pairs

### Experiment: doubling the input again
Watch what a doubling of `n` does this time.

In [ ]:
sizes = [250, 500, 1_000, 2_000, 4_000]
times = []

for n in sizes:
    numbers = list(range(n))
    elapsed = measure(count_pairs, numbers)
    times.append(elapsed)
    print(f"n = {n:>5,}: {elapsed:.4f} seconds")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(sizes, times, marker="o", color="crimson")
plt.title("O(n²): visiting every pair")
plt.xlabel("Size of the list (n)")
plt.ylabel("Time (seconds)")
plt.grid(True)
plt.show()

### Observed result
Each time `n` doubles, the time is multiplied by about **four**. The curve bends upwards. Notice also how small these lists are: the linear experiment handled four million elements comfortably, while here four thousand already take a noticeable time. With four million elements this function would need sixteen million million steps, and would run for days.

## 5. Counting steps instead of seconds

Timing is convincing, but noisy. The cleaner way to reason is to **count the basic steps** as a function of `n`. The table below shows how many steps each class of algorithm needs. The two rows we have not met yet, `log n` and `n log n`, are explained right after.

In [ ]:
import math

print(f"{'n':>12} {'log n':>8} {'n log n':>14} {'n^2':>20}")
for n in [10, 100, 1_000, 1_000_000]:
    print(f"{n:>12,} {math.log2(n):>8.0f} {n * math.log2(n):>14,.0f} {n ** 2:>20,}")

Look at the last row. For a million items:

- a **logarithmic** algorithm needs about **20** steps,
- a **linear** one needs a **million**,
- a **quadratic** one needs a **million million**. At a hundred million steps per second, that is almost three hours.

### O(log n): logarithmic time
An algorithm is O(log n) when each step **cuts the remaining work in half**. The logarithm (base 2) of `n` is simply *the number of times you can halve `n` before reaching 1*. That number grows astonishingly slowly: doubling the input adds just **one** step.

You already know such an algorithm: guessing a number by always choosing the middle of the remaining range (Notebook 08).

In [ ]:
def halving_steps(n):
    steps = 0
    while n > 1:
        n = n // 2          # cut the problem in half
        steps += 1
    return steps

for n in [8, 1_000, 1_000_000, 1_000_000_000]:
    print(f"n = {n:>13,}: {halving_steps(n)} steps")

### O(n log n)
This class sits between linear and quadratic, much closer to linear. It is the cost of the best general sorting algorithms, which we will build in Notebook 14.

### The common classes, from best to worst

| Big O | Name | When n doubles, the time... | Typical example |
|---|---|---|---|
| O(1) | constant | does not change | reading `numbers[i]` |
| O(log n) | logarithmic | grows by one step | binary search |
| O(n) | linear | doubles | summing a list |
| O(n log n) | linearithmic | slightly more than doubles | merge sort |
| O(n²) | quadratic | is multiplied by four | comparing every pair |

In [ ]:
ns = list(range(1, 51))

plt.figure(figsize=(7, 4.5))
plt.plot(ns, [1 for n in ns], label="O(1)")
plt.plot(ns, [math.log2(n) for n in ns], label="O(log n)")
plt.plot(ns, ns, label="O(n)")
plt.plot(ns, [n * math.log2(n) for n in ns], label="O(n log n)")
plt.plot(ns, [n ** 2 for n in ns], label="O(n²)")
plt.ylim(0, 300)
plt.title("How the number of steps grows")
plt.xlabel("Input size (n)")
plt.ylabel("Steps")
plt.legend()
plt.grid(True)
plt.show()

## 6. How to find the Big O by reading code

Two simplification rules make Big O easy to work out.

**Rule 1: ignore constants.** A function that takes `2n` steps and one that takes `n` steps are both **O(n)**. Both double when `n` doubles, and that is all Big O describes.

**Rule 2: keep only the fastest-growing term.** `n² + n + 5` is **O(n²)**. For large `n`, the `n²` part dwarfs everything else.

From these follow the practical recipes:

| Code shape | Big O |
|---|---|
| no loop that depends on n | O(1) |
| one loop over the input | O(n) |
| two loops **one after the other** | O(n + n) = O(n) |
| a loop **inside** a loop, both over the input | O(n × n) = O(n²) |
| a loop that halves the problem at each step | O(log n) |

In [ ]:
# Two loops one AFTER the other: n + n steps  ->  O(n)
def sum_and_count_even(numbers):
    total = 0
    for num in numbers:
        total += num
    even_count = 0
    for num in numbers:
        if num % 2 == 0:
            even_count += 1
    return total, even_count

# A loop INSIDE a loop: n * n steps  ->  O(n^2)
def has_duplicates(numbers):
    for i in range(len(numbers)):
        for j in range(len(numbers)):
            if i != j and numbers[i] == numbers[j]:
                return True
    return False

print(sum_and_count_even([1, 2, 3, 4]))
print(has_duplicates([1, 2, 3, 2]))

### Best case and worst case
`has_duplicates` may stop early, if a duplicate is found at once. That is its **best case**. When there is no duplicate at all, it must check every pair: the **worst case**. Unless stated otherwise, Big O describes the **worst case**, because that is the guarantee we can rely on.

> ⚠️ **Common pitfalls**
>
> - Counting only the visible loops. A call such as `sum(numbers)`, `max(numbers)` or `x in numbers` hides a loop of its own. One of those inside a loop gives O(n²).
> - Believing that fewer lines of code means a faster program. Big O depends on how many steps are executed, not on how many lines are written.

## 7. The cost of Python's own operations

Since built-in operations can hide loops, a programmer must know what they cost.

| Operation | Cost | Why |
|---|---|---|
| `numbers[i]` | O(1) | goes straight to the position |
| `numbers.append(x)` | O(1) | adds at the end, nothing moves |
| `numbers.pop()` | O(1) | removes from the end |
| `numbers.insert(0, x)` | O(n) | every element must shift one place |
| `numbers.pop(0)` | O(n) | every element must shift back |
| `x in numbers` (list) | O(n) | must look at the elements one by one |
| `x in my_dict` or `x in my_set` | O(1) | explained in Notebook 17 |
| `my_dict[key]` | O(1) | explained in Notebook 17 |

### Experiment: `in` on a list versus `in` on a set
A **set** is a collection of unique values, written with curly braces. Its membership test works like a dictionary lookup. Let us search for the same 2,000 values in a list and in a set that hold the same data.

In [ ]:
n = 20_000
as_list = list(range(n))
as_set = set(as_list)
targets = [random.randint(0, n) for i in range(2_000)]

def search_all_in_list(values):
    for value in values:
        value in as_list

def search_all_in_set(values):
    for value in values:
        value in as_set

list_time = measure(search_all_in_list, targets)
set_time = measure(search_all_in_set, targets)

print(f"List: {list_time:.4f} seconds")
print(f"Set:  {set_time:.6f} seconds")
print(f"The set was about {list_time / set_time:,.0f} times faster.")

The same question, the same data, the same answer, and a difference of several hundred times. The only thing that changed is **the data structure**. This single experiment is the reason Module D exists.

---
## ✏️ Exercises

The first exercises follow the worksheet *Determining Big O Complexity and Validating with Experiments*: first decide the complexity by reading the code, then confirm it with a measurement.

### Exercise 1 (Read the code)
Without running anything, state the Big O of each function. Then open the solution.

In [ ]:
def function_a(n):
    return n * 10

def function_b(dictionary, key):
    return dictionary[key]

def function_c(arr):
    count = 0
    for num in arr:
        if num % 2 == 0:
            count += 1
    return count

def function_d(arr):
    return [x for x in arr]

def function_e(arr):
    result = 0
    for i in arr:
        for j in arr:
            result += i + j
    return result
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
# function_a: O(1)    one multiplication, whatever the value of n
# function_b: O(1)    a dictionary lookup does not depend on the size
# function_c: O(n)    one loop over the input
# function_d: O(n)    a comprehension is a loop in disguise
# function_e: O(n^2)  a loop inside a loop, both over the input
print("See the comments above.")
```
</details>

### Exercise 2 (Validate a linear function)
Using `measure`, time `function_c` (counting even numbers) for lists of 500,000, 1,000,000 and 2,000,000 elements, and print the results. Do the times roughly double?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def function_c(arr):
    count = 0
    for num in arr:
        if num % 2 == 0:
            count += 1
    return count

for n in [500_000, 1_000_000, 2_000_000]:
    numbers = list(range(n))
    print(f"n = {n:>9,}: {measure(function_c, numbers):.4f} seconds")
```
</details>

### Exercise 3 (Validate a quadratic function)
Time `function_e` (pairwise addition) for lists of 500, 1,000 and 2,000 elements. By what factor does the time grow at each doubling?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def function_e(arr):
    result = 0
    for i in arr:
        for j in arr:
            result += i + j
    return result

previous = None
for n in [500, 1_000, 2_000]:
    elapsed = measure(function_e, list(range(n)))
    if previous is None:
        print(f"n = {n:>5,}: {elapsed:.4f} seconds")
    else:
        print(f"n = {n:>5,}: {elapsed:.4f} seconds  ({elapsed / previous:.1f} times the previous)")
    previous = elapsed
```

*The factor should be close to 4. Twice the input, four times the work.*
</details>

### Exercise 4 (Count the steps)
Write a function `count_steps(n)` that runs the loops below and returns how many times the innermost line is executed. Call it for `n = 10`, `100` and `1000`. Which Big O is it? (The inner loop does not always run `n` times.)

```python
for i in range(n):
    for j in range(i):
        ...
```

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def count_steps(n):
    steps = 0
    for i in range(n):
        for j in range(i):
            steps += 1
    return steps

for n in [10, 100, 1000]:
    print(n, count_steps(n))
```

*The count is `n(n-1)/2`, about half of `n²`. Half is a constant, and constants are ignored: this is still **O(n²)**. Remember this pattern, it is exactly the shape of the simple sorting algorithms in Notebook 14.*
</details>

### Exercise 5 (A hidden loop)
What is the Big O of the function below? Explain why it is not O(n).

In [ ]:
def count_above_average(numbers):
    count = 0
    for num in numbers:
        if num > sum(numbers) / len(numbers):
            count += 1
    return count
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
# It is O(n^2). The loop runs n times, and on EVERY pass sum(numbers)
# walks through the whole list again: n * n steps.

# The fix: compute the average once, before the loop -> O(n)
def count_above_average_fast(numbers):
    average = sum(numbers) / len(numbers)
    count = 0
    for num in numbers:
        if num > average:
            count += 1
    return count

print(count_above_average_fast([1, 5, 9, 3, 7]))
```

*Moving a calculation that never changes out of a loop is one of the most common and most valuable optimisations.*
</details>

### Exercise 6 (From quadratic to linear)
The function `has_duplicates` in section 6 is O(n²). Write `has_duplicates_fast(numbers)` that gives the same answer in O(n), using a set to remember the values already seen. Then time both on `list(range(3000))`, a list with no duplicates (the worst case).

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def has_duplicates(numbers):
    for i in range(len(numbers)):
        for j in range(len(numbers)):
            if i != j and numbers[i] == numbers[j]:
                return True
    return False

def has_duplicates_fast(numbers):
    seen = set()
    for num in numbers:
        if num in seen:          # O(1) for a set
            return True
        seen.add(num)
    return False

data = list(range(3000))
print(f"Nested loops: {measure(has_duplicates, data):.4f} seconds")
print(f"With a set:   {measure(has_duplicates_fast, data):.6f} seconds")
```
</details>

### Exercise 7 (Insert at the front, a little harder)
The table in section 7 claims that `insert(0, x)` is O(n) while `append(x)` is O(1). Verify it: write two functions, one that builds a list of `n` elements with `append` and one that builds it with `insert(0, ...)`. Time both for `n` = 20,000, 40,000 and 80,000.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def build_with_append(n):
    numbers = []
    for i in range(n):
        numbers.append(i)

def build_with_insert(n):
    numbers = []
    for i in range(n):
        numbers.insert(0, i)

for n in [20_000, 40_000, 80_000]:
    t_append = measure(build_with_append, n)
    t_insert = measure(build_with_insert, n)
    print(f"n = {n:>6,}: append {t_append:.4f} s   insert(0) {t_insert:.4f} s")
```

*Building the list with `append` is O(n) in total. With `insert(0, ...)` each of the `n` insertions costs O(n), so the total is O(n²): the time is multiplied by about four at each doubling. Notebook 16 introduces a structure for which inserting at the front is O(1).*
</details>

## 🔑 Key takeaways

- We judge an algorithm by **how its running time grows** with the input size `n`, not by seconds.
- **O(1)**: the time does not depend on `n`. **O(n)**: it grows in proportion. **O(n²)**: doubling `n` multiplies it by four.
- **O(log n)** appears when each step halves the problem; **O(n log n)** is the cost of good sorting.
- To read the Big O from code: ignore constants, keep the dominant term; consecutive loops **add**, nested loops **multiply**.
- Big O usually describes the **worst case**.
- Built-in operations can hide loops: `in` on a list and `insert(0, x)` are O(n), while dictionary and set lookups are O(1).
- Do not guess: **measure**.

---
**Next:** *Notebook 12: Recursion*, a new way to write repetition, which the fastest algorithms rely on.

---
*© Ioannis Tsioulis. *From Zero to Data Structures & Algorithms in Python*. Prepared for educational use.*